# YOLO Human Pose Estimation

## Introduction
In this project, we utilize YOLO, a state-of-the-art, real-time object detection system, to perform human pose estimation. YOLO provides a robust and efficient solution for detecting and tracking human poses in real-time.

## Objectives
- To implement human pose estimation using YOLO.
- To analyze the accuracy and performance of the YOLO pose estimation model.
- To explore potential applications of human pose estimation in various fields such as sports, healthcare, and entertainment.

## Methodology
1. **Setup and Installation**
    - Install YOLO and other necessary libraries.
    - Set up the environment for running the pose estimation model.

2. **Data Collection**
    - Collect video data or use existing datasets for testing the pose estimation model.
    - Preprocess the data to ensure compatibility with the YOLO framework.

3. **Pose Estimation**
    - Implement the YOLO pose estimation model.
    - Run the model on the collected data to detect and track human poses.

4. **Analysis and Evaluation**
    - Evaluate the accuracy and performance of the pose estimation model.
    - Analyze the results and identify any limitations or areas for improvement.

5. **Applications**
    - Explore potential applications of human pose estimation in various fields.
    - Discuss how the results of this project can be applied in real-world scenarios.

## Implementation
### Setup and Installation


In [1]:
#Install Imports
import subprocess
import sys

# Function to install a package using pip
def install_package(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

# List of required packages
required_packages = [
    "pandas",
    "numpy==1.25",
    "moviepy",
    "matplotlib",
    "seaborn",
    "basemap==1.4.1",
    "opencv-python",
    "ultralytics"
]

# Check and install each package
for package in required_packages:
    try:
        __import__(package)
    except ImportError:
        print(f"Installing {package}...")
        install_package(package)

Installing numpy==1.25...



[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: pip3.11 install --upgrade pip


Installing basemap==1.4.1...



[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: pip3.11 install --upgrade pip


Installing opencv-python...



[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: pip3.11 install --upgrade pip


In [3]:
# Import Required Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from moviepy import VideoFileClip
import cv2
from ultralytics import YOLO

# Set the aesthetic style of the plots
sns.set_style("whitegrid")

In [18]:
# DataFrame to store angles
angles_columns = ['frame', 'left_elbow_angle', 'left_knee_angle', 'left_shoulder_angle', 'left_hip_angle', 'left_ankle_angle',
            'right_elbow_angle', 'right_knee_angle', 'right_shoulder_angle', 'right_hip_angle', 'right_ankle_angle']

joints_columns = ['frame', 'left_elbow', 'left_knee', 'left_shoulder', 'left_hip', 'left_ankle', 'left_wrist','left_feet','right_shoulder',
                   'right_elbow', 'right_wrist', 'right_hip', 'right_knee', 'right_ankle','right_feet']


yolo_joints_df = pd.DataFrame(columns=joints_columns)

yolo_angles_df = pd.DataFrame(columns=angles_columns)

In [5]:
def calculate_angle(a, b, c):
    a = np.array(a)  # First point
    b = np.array(b)  # Mid point
    c = np.array(c)  # End point
    
    radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
    angle = np.abs(radians * 180.0 / np.pi)
    
    if angle > 180.0:
        angle = 360.0 - angle
    
    return angle

In [6]:
model = YOLO('yolo11n-pose.pt')

⚠️ Download failure, retrying 1/3 https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11n-pose.pt...


######################################################################## 100.0%


In [ ]:

# Function to process a single video using YOLO and calculate joint angles
def process_single_video(video_file_path, frame_numbers, yolo_angles_df, yolo_joints_df):
    cap = cv2.VideoCapture(video_file_path)
    
    if not cap.isOpened():
        print(f"Error opening video file {video_file_path}")
        return
    
    print(f"Processing video: {video_file_path}")
    frame_count = 0
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        # Perform pose estimation
        results = model(frame)
        
        keypoints = results[0].keypoints.xy.cpu().numpy()[0]
            
        for keypoint in keypoints:

            cv2.circle(frame, (int(keypoint[0]), int(keypoint[1])), 5, (0, 255, 0), -1)

        left_shoulder = -1
        left_elbow = -1
        left_wrist = -1
        left_hip = -1
        left_knee = -1
        left_ankle = -1
        
        # Get coordinates for right side
        right_shoulder = -1
        right_elbow = -1
        right_wrist = -1
        right_hip = -1
        right_knee = -1
        right_ankle = -1

        # # Calculate angles for left side
        left_elbow_angle = -1
        left_knee_angle = -1
        left_shoulder_angle = -1
        left_hip_angle = -1
        # left_ankle_angle = calculate_angle(left_knee, left_ankle, [left_ankle[0], left_ankle[1] + 0.1])  # Assuming vertical line for ankle
        
        # # Calculate angles for right side
        right_elbow_angle = -1
        right_knee_angle = -1
        right_shoulder_angle = -1
        right_hip_angle = -1
        
        # Extract landmarks
        if len(keypoints) > 0:
            keypoints = [[kp[0],kp[1]] for kp in keypoints]

            """
            0: Nose 1: Left Eye 2: Right Eye 3: Left Ear 4: Right Ear 5: Left Shoulder 6: Right Shoulder 7: Left Elbow 8: Right Elbow 
            9: Left Wrist 10: Right Wrist 11: Left Hip 12: Right Hip 13: Left Knee 14: Right Knee 15: Left Ankle 16: Right Ankle
            """

            print(keypoints)
            # Get coordinates for left side
            left_shoulder = keypoints[5]
            left_elbow = keypoints[7]
            left_wrist = keypoints[9]
            left_hip = keypoints[11]
            left_knee = keypoints[13]
            left_ankle = keypoints[15]
            
            # Get coordinates for right side
            right_shoulder = keypoints[6]
            right_elbow = keypoints[8]
            right_wrist = keypoints[10]
            right_hip = keypoints[12]
            right_knee = keypoints[14]
            right_ankle = keypoints[16]

            # # Calculate angles for left side
            left_elbow_angle = calculate_angle(left_shoulder, left_elbow, left_wrist)
            left_knee_angle = calculate_angle(left_hip, left_knee, left_ankle)
            left_shoulder_angle = calculate_angle(left_hip, left_shoulder, left_elbow)
            left_hip_angle = calculate_angle(left_shoulder, left_hip, left_knee)
            # left_ankle_angle = calculate_angle(left_knee, left_ankle, [left_ankle[0], left_ankle[1] + 0.1])  # Assuming vertical line for ankle
            
            # # Calculate angles for right side
            right_elbow_angle = calculate_angle(right_shoulder, right_elbow, right_wrist)
            right_knee_angle = calculate_angle(right_hip, right_knee, right_ankle)
            right_shoulder_angle = calculate_angle(right_hip, right_shoulder, right_elbow)
            right_hip_angle = calculate_angle(right_shoulder, right_hip, right_knee)
            # right_ankle_angle = calculate_angle(right_knee, right_ankle, [right_ankle[0], right_ankle[1] + 0.1])  # Assuming vertical line for ankle

            for point in [left_shoulder, left_elbow, left_wrist, left_hip, left_knee, left_ankle,
                        right_shoulder, right_elbow, right_wrist, right_hip, right_knee, right_ankle]:
                cv2.circle(frame, tuple(np.multiply(point, [1, 1]).astype(int)), 5, (0, 0, 255), -1)
            # Annotate angles on the image
            cv2.putText(frame, f'Left Elbow: {int(left_elbow_angle)}', tuple(np.multiply(left_elbow, [1, 1]).astype(int)), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2, cv2.LINE_AA)
            cv2.putText(frame, f'Left Knee: {int(left_knee_angle)}', tuple(np.multiply(left_knee, [1, 1]).astype(int)), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2, cv2.LINE_AA)
            cv2.putText(frame, f'Left Shoulder: {int(left_shoulder_angle)}', tuple(np.multiply(left_shoulder, [1, 1]).astype(int)), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2, cv2.LINE_AA)
            cv2.putText(frame, f'Left Hip: {int(left_hip_angle)}', tuple(np.multiply(left_hip, [1, 1]).astype(int)), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2, cv2.LINE_AA)
            
            cv2.putText(frame, f'Right Elbow: {int(right_elbow_angle)}', tuple(np.multiply(right_elbow, [1, 1]).astype(int)), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2, cv2.LINE_AA)
            cv2.putText(frame, f'Right Knee: {int(right_knee_angle)}', tuple(np.multiply(right_knee, [1, 1]).astype(int)), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2, cv2.LINE_AA)
            cv2.putText(frame, f'Right Shoulder: {int(right_shoulder_angle)}', tuple(np.multiply(right_shoulder, [1, 1]).astype(int)), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2, cv2.LINE_AA)
            cv2.putText(frame, f'Right Hip: {int(right_hip_angle)}', tuple(np.multiply(right_hip, [1, 1]).astype(int)), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2, cv2.LINE_AA)
        

        if frame_count in frame_numbers:
                # Get the name of the video
                videoname = video_file_path.split('/')[-1].split('.')[0]
                outdir = "../FINAL/yolo_coco_"+videoname
                # Save the frame with annotated points
                output_frame_path = os.path.join(outdir, f"frame_{frame_count}_pose.jpg")
                cv2.imwrite(output_frame_path, frame)

                # Store points into a DataFrame
                yolo_joints_df = yolo_joints_df.append({
                    'frame': f"frame_{frame_count}",
                    'left_shoulder': left_shoulder,
                    'left_elbow': left_elbow,
                    'left_wrist': left_wrist,
                    'left_hip': left_hip,
                    'left_knee': left_knee,
                    'left_ankle': left_ankle,
                    'right_shoulder': right_shoulder,
                    'right_elbow': right_elbow,
                    'right_wrist': right_wrist,
                    'right_hip': right_hip,
                    'right_knee': right_knee,
                    'right_ankle': right_ankle
                }, ignore_index=True)
            
                yolo_angles_df = yolo_angles_df.append({
                    'frame': f"frame_{frame_count}",
                    'left_elbow_angle': left_elbow_angle,
                    'left_knee_angle': left_knee_angle,
                    'left_shoulder_angle': left_shoulder_angle,
                    'left_hip_angle': left_hip_angle,
                    'right_elbow_angle': right_elbow_angle,
                    'right_knee_angle': right_knee_angle,
                    'right_shoulder_angle': right_shoulder_angle,
                    'right_hip_angle': right_hip_angle
                }, ignore_index=True)
        
                yolo_joints_df.to_json(os.path.join(outdir, 'keypoints.json'), orient='records')
                yolo_angles_df.to_json(os.path.join(outdir, 'angles.json'), orient='records')

        frame_count += 1
        
        # Display the frame
        cv2.imshow('Frame', frame)
        
        # Press Q on keyboard to exit
        if cv2.waitKey(25) & 0xFF == ord('q'):
            break
    
    # Release the video capture object
    cap.release()
    cv2.destroyAllWindows()
    
    return yolo_angles_df

# Function to process videos using YOLO and calculate joint angles
def process_videos_with_yolo(formatted_side_path):
    # Iterate over files in the formatted side directory
    for filename in os.listdir(formatted_side_path):
        if filename.lower().endswith('.mov'):
            base_filename = filename.split('.')[0]
            # Get the list of all frame files in the folder
            frame_folder_path = os.path.join(formatted_side_path, base_filename)
            frame_files = [f for f in os.listdir(frame_folder_path) if f.startswith('frame_') and f.endswith('.jpg')]

            # Extract frame numbers from the filenames and sort them
            frame_numbers = sorted([int(f.split('_')[1].split('.')[0]) for f in frame_files])
            print(frame_numbers)
            video_file_path = os.path.join(formatted_side_path, filename)
            process_single_video(video_file_path, frame_numbers, yolo_angles_df, yolo_joints_df)
    

process_videos_with_yolo("../videos/")

[5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
Processing video: ../videos/maxV_close.MOV

0: 384x640 (no detections), 291.3ms
Speed: 37.8ms preprocess, 291.3ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 254.5ms
Speed: 9.2ms preprocess, 254.5ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 167.0ms
Speed: 3.1ms preprocess, 167.0ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 177.9ms
Speed: 3.9ms preprocess, 177.9ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)
[[0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0]]

0: 384x640 1 person, 141.6ms
Speed: 4.5ms preprocess, 141.6ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)
[[0.0, 0.0], [0.0, 0.0], [0.0, 0.0

/var/folders/j7/tpgdfswn2r31tk28rclvmx4r0000gn/T/ipykernel_77676/3504215946.py:126: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  yolo_joints_df = yolo_joints_df.append({
/var/folders/j7/tpgdfswn2r31tk28rclvmx4r0000gn/T/ipykernel_77676/3504215946.py:142: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  yolo_angles_df = yolo_angles_df.append({
/var/folders/j7/tpgdfswn2r31tk28rclvmx4r0000gn/T/ipykernel_77676/3504215946.py:126: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  yolo_joints_df = yolo_joints_df.append({
/var/folders/j7/tpgdfswn2r31tk28rclvmx4r0000gn/T/ipykernel_77676/3504215946.py:142: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  y


0: 384x640 1 person, 263.1ms
Speed: 3.8ms preprocess, 263.1ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)
[[0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [1509.773, 336.8218], [0.0, 0.0], [1562.9888, 386.19803], [0.0, 0.0], [1697.8939, 381.6009], [0.0, 0.0], [1588.451, 379.19635], [0.0, 0.0], [1607.6279, 577.44635], [1547.4794, 576.2545], [1581.8351, 665.96484], [1445.929, 648.90643], [1811.1954, 789.5539], [1594.5638, 736.0587]]

0: 384x640 1 person, 127.6ms
Speed: 3.3ms preprocess, 127.6ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)


/var/folders/j7/tpgdfswn2r31tk28rclvmx4r0000gn/T/ipykernel_77676/3504215946.py:126: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  yolo_joints_df = yolo_joints_df.append({
/var/folders/j7/tpgdfswn2r31tk28rclvmx4r0000gn/T/ipykernel_77676/3504215946.py:142: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  yolo_angles_df = yolo_angles_df.append({
/var/folders/j7/tpgdfswn2r31tk28rclvmx4r0000gn/T/ipykernel_77676/3504215946.py:126: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  yolo_joints_df = yolo_joints_df.append({
/var/folders/j7/tpgdfswn2r31tk28rclvmx4r0000gn/T/ipykernel_77676/3504215946.py:142: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  y

[[0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [1474.0934, 368.38312], [1435.146, 414.0947], [1552.7333, 379.05432], [0.0, 0.0], [1450.9707, 389.9224], [0.0, 0.0], [1517.952, 540.3217], [1460.7692, 548.2167], [1494.5029, 691.1455], [1299.1693, 691.56116], [1723.8541, 770.3711], [1308.9252, 762.12415]]

0: 384x640 1 person, 203.2ms
Speed: 4.8ms preprocess, 203.2ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)
[[0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [1370.6252, 374.72025], [1333.3827, 415.45908], [1351.2769, 438.4586], [1267.1888, 494.546], [1238.9528, 463.41623], [1225.8025, 483.6892], [1454.8381, 584.8773], [1405.1956, 593.1782], [1373.5437, 693.785], [1210.8748, 707.4855], [1513.4757, 834.6922], [1200.1099, 853.9737]]

0: 384x640 1 person, 125.0ms
Speed: 3.6ms preprocess, 125.0ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)


...

/var/folders/j7/tpgdfswn2r31tk28rclvmx4r0000gn/T/ipykernel_77676/3504215946.py:126: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  yolo_joints_df = yolo_joints_df.append({
/var/folders/j7/tpgdfswn2r31tk28rclvmx4r0000gn/T/ipykernel_77676/3504215946.py:142: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  yolo_angles_df = yolo_angles_df.append({


0: 480x640 1 person, 132.2ms
Speed: 3.0ms preprocess, 132.2ms inference, 1.4ms postprocess per image at shape (1, 3, 480, 640)
[[0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [1550.3005, 650.85297], [1528.8867, 673.39276], [1539.9017, 676.1576], [0.0, 0.0], [1542.9741, 717.7853], [0.0, 0.0], [1568.7994, 740.49335], [1532.1091, 732.33453], [1539.0857, 735.91296], [1569.0813, 783.1837], [1572.3124, 786.12], [1575.3252, 846.30707], [1576.5873, 851.69403]]

0: 480x640 (no detections), 257.7ms
Speed: 43.4ms preprocess, 257.7ms inference, 0.6ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 136.2ms
Speed: 2.8ms preprocess, 136.2ms inference, 0.6ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 162.6ms
Speed: 3.3ms preprocess, 162.6ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)
[[1669.4763, 663.73126], [0.0, 0.0], [1664.9612, 659.26154], [0.0, 0.0], [1658.633, 660.10657], [1673.1848, 681.2986], [1649.3636, 680.7572], [

/var/folders/j7/tpgdfswn2r31tk28rclvmx4r0000gn/T/ipykernel_77676/3504215946.py:126: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  yolo_joints_df = yolo_joints_df.append({
/var/folders/j7/tpgdfswn2r31tk28rclvmx4r0000gn/T/ipykernel_77676/3504215946.py:142: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  yolo_angles_df = yolo_angles_df.append({



0: 480x640 1 person, 144.9ms
Speed: 3.3ms preprocess, 144.9ms inference, 0.9ms postprocess per image at shape (1, 3, 480, 640)
[[0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [1777.801, 677.8929], [1755.8527, 676.9593], [1777.8594, 695.4244], [1724.3673, 685.45715], [1776.2351, 704.4874], [1688.2603, 687.4788], [1775.6188, 743.8632], [1764.3702, 743.22314], [1768.4414, 785.54333], [1768.5521, 784.53955], [1796.5695, 827.8671], [1796.2416, 825.0017]]

0: 480x640 1 person, 182.4ms
Speed: 5.1ms preprocess, 182.4ms inference, 1.1ms postprocess per image at shape (1, 3, 480, 640)
[[0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [1796.7222, 680.8228], [1804.1708, 680.8255], [1784.8306, 705.47736], [1799.633, 702.1588], [1789.2048, 716.58356], [1797.965, 706.9118], [1799.7334, 746.9704], [1804.9669, 748.15356], [1811.981, 790.72894], [1819.4685, 791.80743], [1827.6196, 837.35657], [1830.6559, 837.261]]

0: 480x640 1 person, 155.7ms
Speed: 4.2ms preprocess, 155.7ms in

: 